# 04 — Scaling Laws, Matryoshka SAEs, and Semantic Feature Assignment

This notebook explores three questions:

1. **How does SAE performance scale with dictionary size?** We sweep d_sae from 1x to 16x
   the model dimension and characterize the reconstruction/sparsity tradeoff.

2. **Can we impose feature ordering?** Matryoshka SAEs train features so that the first k
   form a useful reconstruction on their own — recovering the hierarchical ordering that SVD
   gives for free, but with sparse overcomplete features.

3. **How do we systematically assign semantic meaning to features?** We demonstrate three
   complementary methods: max-activating examples, logit lens (vocabulary projection), and
   activation patching (causal intervention).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import time

from sae.model import SparseAutoencoder, MatryoshkaSAE
from sae.activations import load_activations
from sae.analysis import FeatureAnalyzer, logit_lens, activation_patching

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Load cached activations
activations = load_activations("../results/activations_layer6_2M.pt")
print(f"Activations: {activations.shape}")
D_MODEL = activations.shape[1]

---
## Part 1: SAE Dimension Sweep

How does performance change with the number of SAE features (d_sae)?

We expect:
- **More features → lower reconstruction error** (more capacity)
- **More features → more dead features** (harder to utilize all capacity)
- **Diminishing returns** beyond some point

This is analogous to choosing the number of basis functions in a spectral
decomposition — too few and you underfit, too many and you get unused modes.

In [ ]:
def train_sae_quick(d_sae, activations, n_steps=5000, l1_coeff=5e-3, batch_size=4096, lr=1e-3):
    """
    Train an SAE and return final metrics + wall time.
    """
    sae = SparseAutoencoder(d_model=D_MODEL, d_sae=d_sae, l1_coeff=l1_coeff).to(device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)
    dataloader = DataLoader(
        TensorDataset(activations), batch_size=batch_size, shuffle=True, drop_last=True
    )

    sae.train()
    step = 0
    start_time = time.time()

    while step < n_steps:
        for (batch,) in dataloader:
            if step >= n_steps:
                break
            batch = batch.to(device)
            output = sae(batch)
            optimizer.zero_grad()
            output["loss"].backward()
            optimizer.step()
            sae.normalize_decoder()
            step += 1

    train_time = time.time() - start_time

    # Evaluate on held-out data
    sae.eval()
    with torch.no_grad():
        eval_batch = activations[:10000].to(device)
        eval_output = sae(eval_batch)
        features = sae.encode(eval_batch)
        dead = (features.max(dim=0).values == 0).sum().item()

    return {
        "d_sae": d_sae,
        "expansion": d_sae / D_MODEL,
        "mse": eval_output["mse_loss"].item(),
        "l0": eval_output["l0"].item(),
        "loss": eval_output["loss"].item(),
        "dead_features": dead,
        "dead_pct": 100 * dead / d_sae,
        "train_time": train_time,
        "n_params": sum(p.numel() for p in sae.parameters()),
    }

In [ ]:
# Sweep over dictionary sizes
d_sae_values = [768, 1536, 3072, 6144, 12288]
expansion_labels = ["1x", "2x", "4x", "8x", "16x"]

print("Running dimension sweep (5000 steps each)...")
print(f"{'d_sae':>8} {'expand':>6} {'MSE':>10} {'L0':>6} {'Dead%':>7} {'Time(s)':>8} {'Params':>12}")
print("-" * 65)

sweep_results = []
for d_sae in d_sae_values:
    result = train_sae_quick(d_sae, activations)
    sweep_results.append(result)
    print(f"{result['d_sae']:>8} {result['expansion']:>5.0f}x "
          f"{result['mse']:>10.6f} {result['l0']:>6.0f} {result['dead_pct']:>6.1f}% "
          f"{result['train_time']:>7.1f}s {result['n_params']:>12,}")

In [ ]:
# Visualize scaling behavior
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

expansions = [r["expansion"] for r in sweep_results]

# MSE vs expansion
axes[0, 0].plot(expansions, [r["mse"] for r in sweep_results], 'bo-', markersize=8)
axes[0, 0].set_xlabel("Expansion factor (d_sae / d_model)")
axes[0, 0].set_ylabel("Reconstruction MSE")
axes[0, 0].set_title("Reconstruction Quality vs Dictionary Size")
axes[0, 0].set_xscale("log", base=2)
axes[0, 0].set_yscale("log")
axes[0, 0].grid(True, alpha=0.3)

# L0 vs expansion
axes[0, 1].plot(expansions, [r["l0"] for r in sweep_results], 'go-', markersize=8)
axes[0, 1].set_xlabel("Expansion factor")
axes[0, 1].set_ylabel("L0 (avg active features)")
axes[0, 1].set_title("Sparsity vs Dictionary Size")
axes[0, 1].set_xscale("log", base=2)
axes[0, 1].grid(True, alpha=0.3)

# Dead features vs expansion
axes[1, 0].plot(expansions, [r["dead_pct"] for r in sweep_results], 'ro-', markersize=8)
axes[1, 0].set_xlabel("Expansion factor")
axes[1, 0].set_ylabel("Dead features (%)")
axes[1, 0].set_title("Dead Features vs Dictionary Size")
axes[1, 0].set_xscale("log", base=2)
axes[1, 0].grid(True, alpha=0.3)

# Training time vs expansion
axes[1, 1].plot(expansions, [r["train_time"] for r in sweep_results], 'mo-', markersize=8)
axes[1, 1].set_xlabel("Expansion factor")
axes[1, 1].set_ylabel("Training time (s)")
axes[1, 1].set_title("Training Time vs Dictionary Size")
axes[1, 1].set_xscale("log", base=2)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/dimension_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey insight: More features improve reconstruction but increase dead features.")
print("The 8x expansion (6144) is a good balance for GPT-2 small.")

---
## Part 2: Matryoshka SAE

Standard SAEs don't impose any ordering on features — feature 1 is no more
important than feature 6000. This differs from SVD, where singular values
naturally rank components by importance.

**Matryoshka SAEs** recover this ordering by training with a multi-scale loss:
the first k features must reconstruct well on their own, for multiple values of k.
This gives us a sparse overcomplete decomposition WITH a natural hierarchy —
combining the best of SVD (ordering) and SAEs (sparsity, overcompleteness).

This is directly analogous to how in X-ray scattering, the most important
structural modes appear as the largest singular values.

In [ ]:
# Train Matryoshka SAE
TRUNCATION_POINTS = [256, 512, 1024, 2048, 4096, 6144]
D_SAE = 6144
N_STEPS = 10000

print(f"Training Matryoshka SAE: {D_MODEL} → {D_SAE}")
print(f"Truncation points: {TRUNCATION_POINTS}")

matryoshka = MatryoshkaSAE(
    d_model=D_MODEL,
    d_sae=D_SAE,
    l1_coeff=5e-3,
    truncation_points=TRUNCATION_POINTS,
).to(device)

optimizer = torch.optim.Adam(matryoshka.parameters(), lr=1e-3)
dataloader = DataLoader(
    TensorDataset(activations), batch_size=4096, shuffle=True, drop_last=True
)

mat_history = {"loss": [], "mse": [], "prefix_losses": {k: [] for k in TRUNCATION_POINTS}}

matryoshka.train()
step = 0
pbar = tqdm(total=N_STEPS, desc="Matryoshka SAE")

while step < N_STEPS:
    for (batch,) in dataloader:
        if step >= N_STEPS:
            break
        batch = batch.to(device)
        output = matryoshka(batch)

        optimizer.zero_grad()
        output["loss"].backward()
        optimizer.step()
        matryoshka.normalize_decoder()

        if step % 100 == 0:
            mat_history["loss"].append(output["loss"].item())
            mat_history["mse"].append(output["mse_loss"].item())
            for k in TRUNCATION_POINTS:
                mat_history["prefix_losses"][k].append(output["prefix_losses"][k])

        if step % 2000 == 0:
            pbar.set_postfix({"loss": f"{output['loss'].item():.4f}", "L0": f"{output['l0'].item():.0f}"})

        step += 1
        pbar.update(1)

pbar.close()
print(f"Done. Final MSE (full): {mat_history['mse'][-1]:.6f}")

In [ ]:
# Plot prefix reconstruction quality over training
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

steps_logged = np.arange(len(mat_history["loss"])) * 100

# Training curves per prefix
for k in TRUNCATION_POINTS:
    ax1.plot(steps_logged, mat_history["prefix_losses"][k], label=f"k={k}")
ax1.set_xlabel("Training step")
ax1.set_ylabel("MSE")
ax1.set_title("Matryoshka SAE: Reconstruction Loss per Prefix")
ax1.set_yscale("log")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Final MSE vs prefix size
final_prefix_mse = [mat_history["prefix_losses"][k][-1] for k in TRUNCATION_POINTS]
ax2.plot(TRUNCATION_POINTS, final_prefix_mse, 'ro-', markersize=8, label="Matryoshka SAE")
ax2.set_xlabel("Number of features (prefix size k)")
ax2.set_ylabel("Reconstruction MSE")
ax2.set_title("Reconstruction Quality vs Feature Budget")
ax2.set_xscale("log", base=2)
ax2.set_yscale("log")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/matryoshka_training.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Compare: Matryoshka prefix-k vs Standard SAE random-k features
# Load the standard SAE trained in notebook 02
checkpoint = torch.load("../results/sae_gpt2small_layer6.pt", weights_only=False)
standard_sae = SparseAutoencoder(
    d_model=checkpoint["config"]["d_model"],
    d_sae=checkpoint["config"]["d_sae"],
    l1_coeff=checkpoint["config"]["l1_coeff"],
).to(device)
standard_sae.load_state_dict(checkpoint["model_state_dict"])
standard_sae.eval()

# Evaluate both on test data
test_acts = activations[100000:110000].to(device)

# Standard SAE: top-k features by activation magnitude (random order baseline)
with torch.no_grad():
    std_features = standard_sae.encode(test_acts)

# For each k: keep only top-k features by activation, reconstruct
standard_prefix_mse = []
for k in TRUNCATION_POINTS:
    with torch.no_grad():
        # Keep only top-k features by magnitude per sample
        topk_vals, topk_idx = std_features.topk(k, dim=-1)
        masked = torch.zeros_like(std_features)
        masked.scatter_(1, topk_idx, topk_vals)
        recon = standard_sae.decode(masked)
        mse = (test_acts - recon).pow(2).mean().item()
        standard_prefix_mse.append(mse)

# Matryoshka: first-k features (ordered by importance)
matryoshka_prefix_mse = []
with torch.no_grad():
    mat_features = matryoshka.encode(test_acts)
    for k in TRUNCATION_POINTS:
        recon = matryoshka.decode_prefix(mat_features, k)
        mse = (test_acts - recon).pow(2).mean().item()
        matryoshka_prefix_mse.append(mse)

print(f"{'k':>6} {'Matryoshka':>12} {'Std (top-k)':>12} {'Ratio':>8}")
print("-" * 42)
for k, m_mse, s_mse in zip(TRUNCATION_POINTS, matryoshka_prefix_mse, standard_prefix_mse):
    print(f"{k:>6} {m_mse:>12.6f} {s_mse:>12.6f} {m_mse/s_mse:>7.2f}x")

In [ ]:
# Plot the comparison
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(TRUNCATION_POINTS, matryoshka_prefix_mse, 'ro-', markersize=8, linewidth=2,
        label="Matryoshka SAE (first-k features)")
ax.plot(TRUNCATION_POINTS, standard_prefix_mse, 'bs--', markersize=8, linewidth=2,
        label="Standard SAE (top-k by activation)")

ax.set_xlabel("Number of features used (k)", fontsize=12)
ax.set_ylabel("Reconstruction MSE", fontsize=12)
ax.set_title("Matryoshka vs Standard SAE: Anytime Reconstruction", fontsize=13)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/matryoshka_vs_standard.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nMatryoshka SAE gives ordered features: the first k features are always")
print("the best k features for reconstruction — like SVD, but sparse and overcomplete.")

---
## Part 3: Semantic Feature Assignment

How do you figure out what a feature *means*? Three complementary methods:

| Method | What it tells you | Speed | Evidence type |
|--------|-------------------|-------|---------------|
| Max-activating examples | What inputs trigger this feature | Fast | Correlational |
| Logit lens (vocab projection) | What tokens this feature promotes | Very fast | Structural |
| Activation patching | What changes when you remove the feature | Slow | Causal |

Using all three together gives a robust semantic label.

In [ ]:
# Load model for interpretation
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained("gpt2-small", device=device)

# Use our standard SAE (better trained, 15k steps)
sae = standard_sae
tokens = torch.load("../results/tokens_2M.pt", weights_only=True)

### Method A: Max-Activating Examples

The simplest approach: find what inputs most strongly trigger a feature,
then look for patterns. This is what a neuroscientist would do when
characterizing a neuron's receptive field.

In [ ]:
analyzer = FeatureAnalyzer(sae, model=model)

# Get feature statistics to find interesting features
stats = analyzer.feature_statistics(activations[:50000].to(device))
freq = stats["activation_freq"]

# Select 5 features with moderate frequency (interesting and common enough to characterize)
freq_sorted = freq.argsort(descending=True)
target_features = [f.item() for f in freq_sorted if 0.005 < freq[f].item() < 0.1][:5]

print("Selected features for analysis:")
for f in target_features:
    print(f"  Feature {f}: activation frequency = {freq[f].item():.4f}")

In [ ]:
# Show max-activating examples for each feature
for feat_idx in target_features:
    examples = analyzer.display_top_examples(
        feature_idx=feat_idx,
        activations=activations[:200000],
        tokens_flat=tokens[:200000],
        seq_len=128,
        k=10,
        context_window=10,
    )
    
    print(f"\n{'='*70}")
    print(f"Feature {feat_idx} (freq={freq[feat_idx].item():.4f})")
    print(f"{'='*70}")
    for ex in examples:
        display = ""
        for i, t in enumerate(ex["tokens"]):
            if i == ex["target_pos"]:
                display += f" >>>{t}<<< "
            else:
                display += t
        print(f"  [{ex['activation_value']:.2f}] {display.strip()[:120]}")
    print()

### Method B: Logit Lens (Vocabulary Projection)

For each feature, project its decoder direction onto the unembedding matrix.
This tells you what tokens the feature "wants to promote" in the output —
effectively reading the feature's contribution to next-token prediction.

```
logits_i = W_dec[i] @ W_U    (decoder direction × unembedding → vocabulary logits)
```

This is extremely fast (no forward passes needed) and often reveals the
feature's semantic meaning directly.

In [ ]:
# Run logit lens on our target features
vocab_projections = logit_lens(sae, model, feature_indices=target_features, top_k=15)

for feat_idx in target_features:
    result = vocab_projections[feat_idx]
    print(f"\n{'='*70}")
    print(f"Feature {feat_idx} — Vocabulary Projection")
    print(f"{'='*70}")
    
    print(f"\n  TOP tokens (feature PROMOTES these):")
    for token, logit in zip(result["top_tokens"][:10], result["top_logits"][:10]):
        print(f"    {logit.item():+.3f}  '{token}'")
    
    print(f"\n  BOTTOM tokens (feature SUPPRESSES these):")
    for token, logit in zip(result["bottom_tokens"][:5], result["bottom_logits"][:5]):
        print(f"    {logit.item():+.3f}  '{token}'")

### Method C: Activation Patching (Causal Intervention)

The gold standard: remove a feature and measure what changes in the model's
output. High KL divergence = the feature was important for that prediction.

This provides *causal* evidence (not just correlation) that the feature
contributes to specific computations.

In [ ]:
# Test texts for patching
test_texts = [
    "The capital of France is",
    "def fibonacci(n):\n    if n <= 1:\n        return",
    "The cat sat on the",
    "In 2024, the president of the United States",
    "H2O is the chemical formula for",
]

# Patch each target feature on each text
print("Activation Patching Results")
print("="*70)

for feat_idx in target_features[:3]:  # Do 3 features (patching is slower)
    print(f"\nFeature {feat_idx}:")
    print("-" * 50)
    
    for text in test_texts:
        result = activation_patching(sae, model, text, feat_idx, layer=6)
        max_kl = result["kl_divergence"].max().item()
        
        if max_kl > 0.01:  # Only show if feature has meaningful effect
            print(f"  Text: '{text}'")
            print(f"  Max KL divergence: {max_kl:.4f} at token '{result['max_kl_token']}'")
            print(f"  Original prediction: {result['original_top5']['tokens'][:3]}")
            print(f"  Patched prediction:  {result['patched_top5']['tokens'][:3]}")
            print()

### Putting It Together: Assigning Semantic Labels

For each feature, we combine evidence from all three methods:

1. **Max-activating examples** → What contexts trigger this feature?
2. **Logit lens** → What output tokens does this feature promote?
3. **Activation patching** → What computations break when we remove it?

A good semantic label should be consistent across all three:
- If examples are all about code, logit lens shows programming tokens, and
  patching disrupts code completion → label it "Python code" feature.

**The process:**
1. Look at max-activating examples — form initial hypothesis
2. Check logit lens — does vocabulary projection confirm the hypothesis?
3. Test with patching — does removing the feature break relevant predictions?
4. If all three agree → high-confidence semantic label

In [ ]:
# Summary table
print("\n" + "=" * 70)
print("SEMANTIC FEATURE ASSIGNMENT SUMMARY")
print("=" * 70)
print("\nFeature | Freq    | Top Vocab Tokens              | Proposed Label")
print("-" * 70)
for feat_idx in target_features:
    vp = vocab_projections[feat_idx]
    top_toks = ", ".join([f"'{t}'" for t in vp["top_tokens"][:5]])
    print(f"{feat_idx:>7} | {freq[feat_idx].item():.4f} | {top_toks[:35]:35s} | [label manually]")

print("\n→ Fill in the 'Proposed Label' column based on the evidence above.")
print("  A good label is specific and testable (e.g., 'Python function definitions',")
print("  not vague like 'code-related').")

---
## Summary

### Key Findings:

**Scaling:**
- Larger dictionaries monotonically improve reconstruction (lower MSE)
- But dead features increase — the model can't utilize all capacity
- Training time scales linearly with d_sae
- 8x expansion is a practical sweet spot for GPT-2 small

**Matryoshka SAE:**
- Successfully imposes feature ordering without sacrificing full reconstruction quality
- First-k Matryoshka features outperform top-k standard SAE features at small k
- Combines SVD's natural hierarchy with SAE's sparse overcompleteness
- Practical benefit: can use fewer features at inference for speed/memory savings

**Semantic Feature Assignment:**
- Three complementary methods provide robust feature labels
- Logit lens is fastest and often most informative (what does the feature predict?)
- Activation patching provides causal confirmation
- Max-activating examples provide intuitive understanding
- Best practice: use all three and look for convergent evidence

In [ ]:
# Save Matryoshka model
torch.save({
    "model_state_dict": matryoshka.state_dict(),
    "config": {
        "d_model": D_MODEL,
        "d_sae": D_SAE,
        "l1_coeff": 5e-3,
        "truncation_points": TRUNCATION_POINTS,
    },
    "history": mat_history,
}, "../results/matryoshka_sae_gpt2small_layer6.pt")
print("Saved Matryoshka SAE.")
print("Done! All results saved to ../results/")